# 03 – Tiền xử lý dữ liệu (BRFSS 2015)
*Phụ trách: TV1, TV2.* Chỉ dùng pandas, numpy, scikit-learn.

**Quyết định**
1. **Dòng trùng lặp:** xóa (giữ bản đầu) **trước** khi chia train/test, để cùng một hồ sơ không nằm ở cả train và test (rò rỉ). Tác động: bỏ 23.899 dòng (9,42%), tỷ lệ lớp 0 giảm 84,2% → 82,7%. Đánh đổi: nhiều dòng trùng là người khác nhau có cùng hồ sơ, nên nhóm nên so sánh điểm mô hình khi có/không xóa trùng.
2. **Ngoại lệ BMI:** winsorization theo phân vị 1%–99%, ước lượng chỉ trên train (nằm trong Pipeline). Không xóa dòng vì phần lớn là béo phì thật. MentHlth, PhysHlth bị chặn trong [0, 30] nên giữ nguyên.
3. **Biến thứ tự** (Age, Education, Income, GenHlth) giữ dạng số; có tuỳ chọn one-hot cho mô hình tuyến tính.
4. **StandardScaler** nằm trong Pipeline, chỉ fit trên train.
5. Lưu dữ liệu đã bỏ trùng vào `DATA/processed/` cho các notebook sau.

In [1]:
import os
import numpy as np, pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# ---- Cấu hình (đường dẫn tính từ notebooks/03_Data_Preprocessing/) ----
RAW_DIR = "../../DATA/raw"
PROCESSED_DIR = "../../DATA/processed"
RAW_FILE = "diabetes.csv"
TARGET = "Diabetes_binary"
ORDINAL = ["Age", "Education", "Income", "GenHlth"]
SEED = 42
os.makedirs(PROCESSED_DIR, exist_ok=True)

## Các hàm

In [2]:
def load_raw(raw_file=RAW_FILE):
    """Đọc file thô từ DATA/raw."""
    return pd.read_csv(os.path.join(RAW_DIR, raw_file))


def drop_duplicates(df):
    """Xóa dòng trùng (giữ bản đầu). Trả về (df sạch, dict thống kê)."""
    out = df.drop_duplicates(keep="first").reset_index(drop=True)
    info = {"rows_before": len(df), "rows_after": len(out), "removed": len(df) - len(out),
            "removed_pct": round((len(df) - len(out)) / len(df) * 100, 2)}
    return out, info


def split_train_test(df, target=TARGET, test_size=0.2, seed=SEED):
    """Chia train/test có phân tầng theo lớp mục tiêu."""
    X, y = df.drop(columns=target), df[target].astype(int)
    return train_test_split(X, y, test_size=test_size, stratify=y, random_state=seed)


class Winsorizer(BaseEstimator, TransformerMixin):
    """Cắt các cột được chọn về [phân vị lower, phân vị upper]; cận chỉ ước lượng khi fit (trên train)."""
    def __init__(self, columns=("BMI",), lower=0.01, upper=0.99):
        self.columns, self.lower, self.upper = columns, lower, upper

    def fit(self, X, y=None):
        X = pd.DataFrame(X)
        self.bounds_ = {c: (X[c].quantile(self.lower), X[c].quantile(self.upper)) for c in self.columns}
        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()
        for c, (lo, hi) in self.bounds_.items():
            X[c] = X[c].clip(lo, hi)
        return X


def build_pipeline(model, onehot_ordinal=False, winsorize_bmi=True):
    """Pipeline: [Winsorizer BMI] -> [one-hot biến thứ tự] -> StandardScaler -> model."""
    steps = []
    if winsorize_bmi:
        steps.append(("winsor", Winsorizer(("BMI",))))
    if onehot_ordinal:
        ct = ColumnTransformer(
            [("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False, drop="first"), ORDINAL)],
            remainder="passthrough", verbose_feature_names_out=False)
        ct.set_output(transform="pandas")
        steps.append(("onehot", ct))
    steps += [("scaler", StandardScaler()), ("model", model)]
    return Pipeline(steps)


def duplicate_leak_check(raw_df, target=TARGET, seed=SEED):
    """% dòng test có hồ sơ (21 đặc trưng) giống hệt một dòng train: khi không xóa trùng vs đã xóa trùng."""
    key = lambda X: X.astype(int).astype(str).agg("-".join, axis=1)
    res = {}
    for name, d in [("không xóa trùng", raw_df), ("đã xóa trùng", raw_df.drop_duplicates())]:
        Xtr, Xte, _, _ = split_train_test(d.reset_index(drop=True), target, seed=seed)
        res[name] = f"{key(Xte).isin(set(key(Xtr))).mean():.2%}"
    return res

## 3.1 Xử lý dòng trùng lặp và lưu dữ liệu

In [3]:
raw = load_raw()
df, info = drop_duplicates(raw)
print("Bỏ trùng:", info)
print("Phân bố lớp trước:", raw[TARGET].value_counts(normalize=True).round(4).to_dict())
print("Phân bố lớp sau  :", df[TARGET].value_counts(normalize=True).round(4).to_dict())
print("% dòng test trùng hồ sơ trong train:", duplicate_leak_check(raw))

df.to_csv(os.path.join(PROCESSED_DIR, "diabetes_dedup.csv"), index=False)
print("Đã lưu:", os.path.join(PROCESSED_DIR, "diabetes_dedup.csv"))

Bỏ trùng: {'rows_before': 253680, 'rows_after': 229474, 'removed': 24206, 'removed_pct': 9.54}
Phân bố lớp trước: {0: 0.8607, 1: 0.1393}
Phân bố lớp sau  : {0: 0.8471, 1: 0.1529}
% dòng test trùng hồ sơ trong train: {'không xóa trùng': '13.47%', 'đã xóa trùng': '1.09%'}
Đã lưu: ../../DATA/processed\diabetes_dedup.csv


**Nhận xét.** Bỏ 23.899 dòng (9,42%), gần như toàn bộ thuộc lớp 0 nên tỷ lệ lớp 0 giảm từ 84,2% xuống 82,7%, còn lớp tiền ĐTĐ và ĐTĐ tăng nhẹ (2,0% và 15,3%). Nếu giữ nguyên, 13,65% dòng test có hồ sơ y hệt một dòng train (nguy cơ rò rỉ làm điểm test cao giả); sau khi xóa còn 1,31% (cùng đặc trưng nhưng khác nhãn nên không bị coi là trùng).

## 3.2 Chia train/test và kiểm tra Pipeline
Kiểm tra scaler không nhìn thấy test: trung bình train ≈ 0 còn trung bình test lệch khỏi 0 một chút.

In [4]:
Xtr, Xte, ytr, yte = split_train_test(df)
print("Train/Test:", Xtr.shape, Xte.shape)
print("Lớp train:", ytr.value_counts(normalize=True).round(4).to_dict())
print("Lớp test :", yte.value_counts(normalize=True).round(4).to_dict())

for oh in (False, True):
    pipe = build_pipeline(LogisticRegression(max_iter=1000), onehot_ordinal=oh).fit(Xtr, ytr)
    Ztr, Zte = pipe[:-1].transform(Xtr), pipe[:-1].transform(Xte)
    print(f"\nonehot_ordinal={oh}: số đặc trưng = {Ztr.shape[1]}")
    print(f"  max |mean| train = {np.abs(Ztr.mean(0)).max():.1e} | max |mean| test = {np.abs(Zte.mean(0)).max():.3f}")
    print("  Cận winsorization BMI (từ train):", pipe.named_steps["winsor"].bounds_)

Train/Test: (183579, 21) (45895, 21)
Lớp train: {0: 0.8471, 1: 0.1529}
Lớp test : {0: 0.8471, 1: 0.1529}

onehot_ordinal=False: số đặc trưng = 21
  max |mean| train = 3.3e-16 | max |mean| test = 0.012
  Cận winsorization BMI (từ train): {'BMI': (np.float64(18.0), np.float64(50.0))}

onehot_ordinal=True: số đặc trưng = 45
  max |mean| train = 3.0e-16 | max |mean| test = 0.012
  Cận winsorization BMI (từ train): {'BMI': (np.float64(18.0), np.float64(50.0))}


**Nhận xét.** Chia train/test 80/20 có phân tầng giữ nguyên tỷ lệ lớp ở cả hai tập. Scaler chỉ fit trên train nên trung bình train ≈ 0 (~1e-16) còn test lệch nhẹ (~0,01–0,02), đúng như mong đợi nếu không có rò rỉ. Cận winsorization của BMI là [18, 50]. One-hot các biến thứ tự tăng số đặc trưng từ 21 lên 45. Bước chia train/test cuối cùng và việc lưu train/test sẽ làm ở notebook 05.